# The same twenty lines, in three frameworks

A linear classifier trained from scratch in TensorFlow, PyTorch, and JAX. Reading the three side by side is the fastest way to see what is essential and what is dialect.

**Runs on:** CPU — 2 minutes. Only the backend you have installed will run; read the others. &nbsp;·&nbsp; **Slides:** [Chapter 3 — Introduction to TensorFlow, PyTorch, JAX, and Keras](../../../course-web-slides/ch03/index.html) &nbsp;·&nbsp; **Section:** 02 — The three frameworks

---

## The problem, once

In [ ]:
import numpy as np

num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(
    mean=[0, 3], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class)
positive_samples = np.random.multivariate_normal(
    mean=[3, 0], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class)

inputs = np.vstack((negative_samples, positive_samples)).astype("float32")
targets = np.vstack((np.zeros((num_samples_per_class, 1), dtype="float32"),
                     np.ones((num_samples_per_class, 1), dtype="float32")))

import matplotlib.pyplot as plt
plt.figure(figsize=(5, 5))
plt.scatter(inputs[:, 0], inputs[:, 1], c=targets[:, 0], s=8, cmap="coolwarm")
plt.gca().set_aspect("equal"); plt.title("Two Gaussian blobs"); plt.show()

## TensorFlow: a tape, and assign_sub

In [ ]:
import tensorflow as tf

W = tf.Variable(initial_value=tf.random.uniform(shape=(2, 1)))
b = tf.Variable(initial_value=tf.zeros(shape=(1,)))

def model(x):
    return tf.matmul(x, W) + b

def mean_squared_error(targets, predictions):
    return tf.reduce_mean(tf.square(targets - predictions))

learning_rate = 0.1

def training_step(inputs, targets):
    with tf.GradientTape() as tape:
        predictions = model(inputs)
        loss = mean_squared_error(targets, predictions)
    grad_loss_wrt_W, grad_loss_wrt_b = tape.gradient(loss, [W, b])
    W.assign_sub(grad_loss_wrt_W * learning_rate)
    b.assign_sub(grad_loss_wrt_b * learning_rate)
    return loss

for step in range(40):
    loss = training_step(inputs, targets)
    if step % 10 == 0:
        print(f"step {step:3d}  loss {float(loss):.4f}")

Expected output:

```
step   0  loss 3.xxxx
step  10  loss 0.0xxx
step  20  loss 0.0xxx
step  30  loss 0.0xxx
```

## PyTorch: backward(), and no_grad()

In [ ]:
import torch

W_t = torch.rand(2, 1, requires_grad=True)
b_t = torch.zeros(1, requires_grad=True)

x_t = torch.tensor(inputs)
y_t = torch.tensor(targets)

for step in range(40):
    predictions = torch.matmul(x_t, W_t) + b_t
    loss = torch.mean(torch.square(y_t - predictions))
    loss.backward()                     # gradients accumulate onto .grad
    with torch.no_grad():               # the update is not part of the graph
        W_t -= W_t.grad * 0.1
        b_t -= b_t.grad * 0.1
        W_t.grad.zero_()                # ...and must be cleared, every step
        b_t.grad.zero_()
    if step % 10 == 0:
        print(f"step {step:3d}  loss {float(loss):.4f}")

> ⚠️ **The two lines people forget.** `no_grad()` around the update, and `grad.zero_()` after it. Omit the second and gradients accumulate across steps — the model still trains, badly, with no error.

## JAX: no state at all

In [ ]:
import jax
import jax.numpy as jnp

def compute_loss(state, inputs, targets):
    W, b = state
    predictions = jnp.matmul(inputs, W) + b
    return jnp.mean(jnp.square(targets - predictions))

grad_fn = jax.jit(jax.value_and_grad(compute_loss))

state = (jnp.array(np.random.uniform(size=(2, 1)), dtype="float32"),
         jnp.zeros((1,), dtype="float32"))

for step in range(40):
    loss, grads = grad_fn(state, inputs, targets)
    state = tuple(p - g * 0.1 for p, g in zip(state, grads))
    if step % 10 == 0:
        print(f"step {step:3d}  loss {float(loss):.4f}")

No variables, no in-place mutation, no `.grad` to clear. `value_and_grad` turns a **function** into another function, and `jit` compiles it. Chapter 18 recommends JAX for distributed training, and this slide is why: there is no hidden state to shard.

## What was the same

| | TensorFlow | PyTorch | JAX |
|---|---|---|---|
| gradients | `tape.gradient` | `loss.backward()` | `jax.grad` |
| state | `tf.Variable` | tensors with `requires_grad` | **none — passed in** |
| update | `assign_sub` | in place under `no_grad` | build a new tuple |
| clearing | not needed | **`grad.zero_()`** | not applicable |

The forward pass and the loss are identical in all three. **Everything that differs is state management.**

---

## What to take away

- The maths is the same in all three; the differences are entirely about where state lives.
- TensorFlow records on a tape; PyTorch accumulates onto `.grad`; JAX transforms pure functions.
- PyTorch's `zero_()` is a silent-failure trap worth knowing before you meet it.
- JAX's statelessness is the reason chapter 18 prefers it for distributed training.